# Causal Multipart RVQ-VAE Validation

This notebook validates the four causal body codecs used by Phase 1. It answers five separate questions:

1. **Strict causality:** do prefix and one-shot execution produce identical RVQ IDs, and can future inputs/tokens change the past?
2. **Reconstruction quality:** how well are raw features, velocities, accelerations, 6D rotations, and root trajectory preserved?
3. **RVQ value:** does adding q1, q2, and q3 improve over q0 alone?
4. **Codebook health:** how many codes are used, what is the entropy/effective vocabulary, and does one code dominate?
5. **Runtime:** are encoding and decoding faster than real time?

Face is intentionally excluded. Run Jupyter with `NVIDIA_TF32_OVERRIDE=0` for canonical prefix-stable RVQ decisions.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'motion_generation').is_dir() and (candidate / 'SuSuInterActs').exists():
            return candidate
    raise RuntimeError('Could not locate the sentiAvatar-sandbox project root')

PROJECT_DIR = find_project_root()
MOTION_GENERATION_DIR = PROJECT_DIR / 'motion_generation'
if str(MOTION_GENERATION_DIR) not in sys.path:
    sys.path.insert(0, str(MOTION_GENERATION_DIR))

from scripts.audit_causal_body_codecs import audit_part_clip
from scripts.export_multipart_motion_tokens import configure_strict_inference_math, load_part_codec
from utils.causal_codec_evaluation import evaluate_dataset
from utils.multipart_motion import PART_ORDER, load_motion_dict, load_name_list, motion_path_for_name, split_motion_parts

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
print('PROJECT_DIR:', PROJECT_DIR)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Configuration

The full reconstruction pass uses all 635 validation clips by default. The RVQ-level ablation is deliberately smaller because it performs four decodes per part. Set `MAX_EVAL_CLIPS=32` for an initial smoke run.

In [ ]:
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DATA_DIR = PROJECT_DIR / 'SuSuInterActs' / 'SuSuInterActs'
MOTION_DIR = DATA_DIR / 'motion_data'
VAL_SPLIT = DATA_DIR / 'split' / 'val_file_list.txt'
CKPT_ROOT = PROJECT_DIR / 'checkpoints' / 'causal_multipart_rvqvae'
CHECKPOINTS = {
    'upper': CKPT_ROOT / 'causal_rvq_upper_512x4_scratch' / 'model' / 'best.pth',
    'lower': CKPT_ROOT / 'causal_rvq_lower_512x4_scratch' / 'model' / 'best.pth',
    'feet': CKPT_ROOT / 'causal_rvq_feet_512x4_scratch' / 'model' / 'best.pth',
    'hands': CKPT_ROOT / 'causal_rvq_hands_512x4_scratch' / 'model' / 'best.pth',
}
OUTPUT_DIR = PROJECT_DIR / 'motion_generation' / 'outputs' / 'causal_codec_validation'

MAX_EVAL_CLIPS = None       # None = all 635 validation clips
RUN_CAUSAL_AUDIT = True
CAUSAL_AUDIT_CLIPS = 8
CAUSAL_MAX_SOURCE_FRAMES = 128
CAUSAL_ATOL = 0.005
RUN_RVQ_ABLATION = True
RVQ_ABLATION_CLIPS = 128
QUALITATIVE_EXAMPLES = 3
ROOT_ABS_THRESHOLD = 10.0

names = load_name_list(VAL_SPLIT)
if MAX_EVAL_CLIPS is not None:
    names = names[:MAX_EVAL_CLIPS]
print('DEVICE:', DEVICE)
print('Validation clips selected:', len(names))
print('NVIDIA_TF32_OVERRIDE:', os.environ.get('NVIDIA_TF32_OVERRIDE'))

In [ ]:
path_rows = [
    {'item': 'data_dir', 'path': str(DATA_DIR), 'exists': DATA_DIR.is_dir()},
    {'item': 'val_split', 'path': str(VAL_SPLIT), 'exists': VAL_SPLIT.is_file()},
] + [
    {'item': f'{part}_checkpoint', 'path': str(path), 'exists': path.is_file()}
    for part, path in CHECKPOINTS.items()
]
path_table = pd.DataFrame(path_rows)
display(path_table)
missing = path_table.loc[~path_table['exists']]
assert missing.empty, f'Missing prerequisites:\n{missing.to_string(index=False)}'

## Load the four final checkpoints

TF32 is disabled programmatically before checkpoint inference. The process-level environment override is still recommended.

In [ ]:
math_mode = configure_strict_inference_math(DEVICE)
codecs = {part: load_part_codec(CHECKPOINTS[part], DEVICE) for part in PART_ORDER}
codec_rows = []
for part, loaded in codecs.items():
    codec_rows.append({
        'part': part,
        'causal': loaded.causal,
        'unit_length': loaded.unit_length,
        'token_fps': 20.0 / loaded.unit_length,
        'codebook_size': loaded.codebook_size,
        'num_quantizers': loaded.num_quantizers,
        'checkpoint': str(loaded.checkpoint_path),
        'normalizer': str(loaded.normalizer_path),
    })
codec_table = pd.DataFrame(codec_rows)
display(codec_table)
print('Strict math mode:', json.dumps(math_mode, indent=2))
assert codec_table['causal'].all(), 'At least one checkpoint is not causal'
assert set(codec_table['unit_length']) == {2}, 'Expected one 10 Hz token per two 20 Hz frames'

## Gate 1 — strict causal behavior

This is a non-negotiable correctness gate. Continuous tensors use `CAUSAL_ATOL`, while all RVQ IDs must match exactly. The audit also perturbs future motion and future tokens.

In [ ]:
causal_rows = []
if RUN_CAUSAL_AUDIT:
    audited = 0
    for name in tqdm(names, desc='Causal audit candidates'):
        motion_path = motion_path_for_name(MOTION_DIR, name)
        if not motion_path.is_file():
            continue
        parts, _ = split_motion_parts(load_motion_dict(motion_path), abs_threshold=ROOT_ABS_THRESHOLD)
        if min(value.shape[0] for value in parts.values()) < 4:
            continue
        for part in PART_ORDER:
            result = audit_part_clip(
                codecs[part], parts[part], DEVICE,
                max_source_frames=CAUSAL_MAX_SOURCE_FRAMES, atol=CAUSAL_ATOL,
            )
            causal_rows.append({'name': name, 'part': part, 'status': 'passed', **result})
        audited += 1
        if audited >= CAUSAL_AUDIT_CLIPS:
            break
    assert audited == min(CAUSAL_AUDIT_CLIPS, len(names)), f'Only audited {audited} clips'
causal_df = pd.DataFrame(causal_rows)
display(causal_df)
print(f'PASS: {len(causal_df)} clip-part causal audits')

## Full four-level reconstruction evaluation

Metrics are computed after denormalization. Raw feature RMSE is useful within a part but should not be compared directly across parts with different feature scales. Rotation geodesic error is the most interpretable cross-part pose metric.

In [ ]:
full_result = evaluate_dataset(
    codecs=codecs, data_dir=DATA_DIR, names=names, device=DEVICE,
    rvq_levels=(4,), max_clips=None, max_examples=QUALITATIVE_EXAMPLES,
    root_abs_threshold=ROOT_ABS_THRESHOLD,
)
full_df = pd.DataFrame(full_result['rows'])
usage_df = pd.DataFrame(full_result['usage_rows'])
error_df = pd.DataFrame(full_result['errors'])
print('Evaluated:', full_result['evaluated_clips'], '/', full_result['requested_clips'])
print('Root schemas:', full_result['root_schemas'])
if not error_df.empty:
    display(error_df.head(20))
assert error_df.empty, f"{len(error_df)} validation clips failed"
assert np.isfinite(full_df['feature_rmse']).all(), 'Non-finite reconstruction metrics'
display(full_df.head())

In [ ]:
summary_columns = [
    'normalized_smooth_l1', 'normalized_velocity_smooth_l1',
    'feature_mae', 'feature_rmse', 'velocity_rmse', 'acceleration_rmse',
    'geodesic_deg_mean', 'geodesic_deg_p90', 'encode_rtf', 'decode_rtf',
]
summary_rows = []
for part, group in full_df.groupby('part', sort=False):
    row = {'part': part, 'clips': group['name'].nunique(), 'hours': group['duration_seconds'].sum() / 3600.0}
    for metric in summary_columns:
        row[f'{metric}_mean'] = group[metric].mean()
        row[f'{metric}_median'] = group[metric].median()
        row[f'{metric}_p90'] = group[metric].quantile(0.90)
    if part == 'lower':
        for metric in ['root_velocity_rmse', 'root_trajectory_error_mean', 'root_trajectory_final_drift']:
            row[f'{metric}_mean'] = group[metric].mean()
            row[f'{metric}_p90'] = group[metric].quantile(0.90)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_table = full_df.groupby('part', sort=False).mean(numeric_only=True).reindex(PART_ORDER)
plot_table['geodesic_deg_mean'].plot.bar(ax=axes[0], color='#4C78A8', title='Mean rotation geodesic error')
axes[0].set_ylabel('degrees')
plot_table['velocity_rmse'].plot.bar(ax=axes[1], color='#F58518', title='Mean raw velocity RMSE')
axes[1].set_ylabel('raw feature units/frame')
plot_table[['encode_rtf', 'decode_rtf']].plot.bar(ax=axes[2], title='Compute real-time factor')
axes[2].axhline(1.0, color='red', linestyle='--', linewidth=1, label='real time')
axes[2].legend()
for axis in axes:
    axis.tick_params(axis='x', rotation=0)
    axis.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Codebook health

Utilization alone is not enough: a codec can touch many codes while concentrating most tokens in a few entries. Inspect utilization, entropy/effective codes, and top-code share together for every part and quantizer.

In [ ]:
display(usage_df)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for metric, axis, title in [
    ('utilization', axes[0], 'Codebook utilization'),
    ('effective_codes', axes[1], 'Effective number of codes'),
    ('top_code_share', axes[2], 'Most frequent code share'),
]:
    usage_df.pivot(index='part', columns='quantizer', values=metric).reindex(PART_ORDER).plot.bar(ax=axis, title=title)
    axis.tick_params(axis='x', rotation=0)
    axis.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## RVQ-level ablation: q0 versus q0–q3

A healthy residual quantizer should usually improve reconstruction as levels are added. This does not require every individual clip to be monotonic, but the aggregate q0–q3 curve should improve materially.

In [ ]:
ablation_df = pd.DataFrame()
ablation_summary = pd.DataFrame()
if RUN_RVQ_ABLATION:
    ablation_result = evaluate_dataset(
        codecs=codecs, data_dir=DATA_DIR, names=names, device=DEVICE,
        rvq_levels=(1, 2, 3, 4), max_clips=min(RVQ_ABLATION_CLIPS, len(names)),
        max_examples=0, root_abs_threshold=ROOT_ABS_THRESHOLD,
    )
    assert not ablation_result['errors'], ablation_result['errors'][:10]
    ablation_df = pd.DataFrame(ablation_result['rows'])
    ablation_summary = ablation_df.groupby(['part', 'rvq_levels'], as_index=False).agg(
        clips=('name', 'nunique'),
        feature_rmse=('feature_rmse', 'mean'),
        velocity_rmse=('velocity_rmse', 'mean'),
        geodesic_deg_mean=('geodesic_deg_mean', 'mean'),
        normalized_smooth_l1=('normalized_smooth_l1', 'mean'),
    )
    display(ablation_summary)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for part in PART_ORDER:
        part_df = ablation_summary[ablation_summary['part'] == part]
        axes[0].plot(part_df['rvq_levels'], part_df['feature_rmse'], marker='o', label=part)
        axes[1].plot(part_df['rvq_levels'], part_df['velocity_rmse'], marker='o', label=part)
        axes[2].plot(part_df['rvq_levels'], part_df['geodesic_deg_mean'], marker='o', label=part)
    axes[0].set_title('Feature RMSE by RVQ levels')
    axes[1].set_title('Velocity RMSE by RVQ levels')
    axes[2].set_title('Geodesic error by RVQ levels')
    for axis in axes:
        axis.set_xlabel('levels used (1=q0, 4=q0–q3)')
        axis.set_xticks([1, 2, 3, 4])
        axis.grid(alpha=0.25)
    axes[0].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
refinement_rows = []
if not ablation_summary.empty:
    for part in PART_ORDER:
        part_df = ablation_summary[ablation_summary['part'] == part].set_index('rvq_levels')
        refinement_rows.append({
            'part': part,
            'feature_rmse_reduction_q0_to_q3': 1.0 - part_df.loc[4, 'feature_rmse'] / part_df.loc[1, 'feature_rmse'],
            'velocity_rmse_reduction_q0_to_q3': 1.0 - part_df.loc[4, 'velocity_rmse'] / part_df.loc[1, 'velocity_rmse'],
            'geodesic_reduction_q0_to_q3': 1.0 - part_df.loc[4, 'geodesic_deg_mean'] / part_df.loc[1, 'geodesic_deg_mean'],
            'q3_not_worse_feature_rmse': bool(part_df.loc[4, 'feature_rmse'] <= part_df.loc[1, 'feature_rmse']),
        })
refinement_df = pd.DataFrame(refinement_rows)
display(refinement_df)

## Qualitative error timelines

These plots reveal short bursts and drift that clip-level averages can hide. The lower-body second panel shows integrated root displacement; other parts show representative raw feature channels.

In [ ]:
for name, example in full_result['examples'].items():
    fig, axes = plt.subplots(len(PART_ORDER), 2, figsize=(15, 12))
    fig.suptitle(name, fontsize=12)
    for row, part in enumerate(PART_ORDER):
        target = example['target'][part]
        prediction = example['prediction'][part]
        frame_rmse = np.sqrt(np.mean((prediction - target) ** 2, axis=-1))
        axes[row, 0].plot(frame_rmse, color='#D62728')
        axes[row, 0].set_title(f'{part}: per-frame feature RMSE')
        axes[row, 0].set_xlabel('20 Hz frame')
        axes[row, 0].grid(alpha=0.25)
        if part == 'lower':
            axes[row, 1].plot(np.cumsum(target[:, 0]), label='GT root x')
            axes[row, 1].plot(np.cumsum(prediction[:, 0]), label='reconstructed root x')
            axes[row, 1].set_title('lower: integrated root x')
        else:
            for feature in range(min(3, target.shape[1])):
                axes[row, 1].plot(target[:, feature], alpha=0.75, label=f'GT d{feature}')
                axes[row, 1].plot(prediction[:, feature], '--', alpha=0.75, label=f'rec d{feature}')
            axes[row, 1].set_title(f'{part}: representative raw features')
        axes[row, 1].legend(ncol=2, fontsize=8)
        axes[row, 1].grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## Decision gates and saved report

The invariant gates below are intentionally strict. Absolute quality thresholds are not hard-coded because raw part scales differ; acceptability should be decided from the reported geodesic error, root drift, q0→q3 gain, and visual examples.

In [ ]:
gates = [
    {'gate': 'all selected clips evaluated', 'passed': full_result['evaluated_clips'] == full_result['requested_clips']},
    {'gate': 'no reconstruction failures', 'passed': len(full_result['errors']) == 0},
    {'gate': 'all reconstruction metrics finite', 'passed': bool(np.isfinite(full_df['feature_rmse']).all())},
    {'gate': 'all four parts faster than real time (encode)', 'passed': bool((full_df.groupby('part')['encode_rtf'].mean() < 1.0).all())},
    {'gate': 'all four parts faster than real time (decode)', 'passed': bool((full_df.groupby('part')['decode_rtf'].mean() < 1.0).all())},
]
if RUN_CAUSAL_AUDIT:
    gates.insert(0, {'gate': 'strict causal audit completed', 'passed': len(causal_df) == CAUSAL_AUDIT_CLIPS * len(PART_ORDER)})
if not refinement_df.empty:
    gates.append({'gate': 'q0–q3 feature RMSE no worse than q0', 'passed': bool(refinement_df['q3_not_worse_feature_rmse'].all())})
gate_df = pd.DataFrame(gates)
display(gate_df)
assert gate_df['passed'].all(), gate_df.loc[~gate_df['passed']].to_dict('records')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
full_df.to_csv(OUTPUT_DIR / 'per_clip_part_metrics.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'summary_by_part.csv', index=False)
usage_df.to_csv(OUTPUT_DIR / 'codebook_usage.csv', index=False)
causal_df.to_csv(OUTPUT_DIR / 'causal_audit.csv', index=False)
if not ablation_df.empty:
    ablation_df.to_csv(OUTPUT_DIR / 'rvq_level_ablation_per_clip.csv', index=False)
    ablation_summary.to_csv(OUTPUT_DIR / 'rvq_level_ablation_summary.csv', index=False)
report = {
    'status': 'passed', 'device': str(DEVICE), 'math_mode': math_mode,
    'validation_clips': full_result['evaluated_clips'],
    'root_schemas': full_result['root_schemas'],
    'checkpoints': {part: str(path) for part, path in CHECKPOINTS.items()},
    'gates': gate_df.to_dict('records'),
    'summary_by_part': summary_df.to_dict('records'),
    'codebook_usage': usage_df.to_dict('records'),
    'rvq_refinement': refinement_df.to_dict('records'),
}
(OUTPUT_DIR / 'report.json').write_text(json.dumps(report, indent=2, allow_nan=True), encoding='utf-8')
print('PASS: causal codec validation complete')
print('Artifacts:', OUTPUT_DIR)

## How to interpret the result

- **Any causal audit failure is a blocker.** Exact token-prefix equality and future independence must pass with TF32 disabled.
- `normalized_smooth_l1` and `normalized_velocity_smooth_l1` correspond most closely to the training objectives.
- `geodesic_deg_mean/p90` measure actual rotational disagreement and are more comparable across parts than raw RMSE.
- Lower-body root trajectory error reveals accumulated drift that framewise velocity RMSE can hide.
- q0→q3 curves should improve overall. Flat later levels suggest unused capacity; regression suggests a quantization/decoder problem.
- Low utilization is not automatically failure, but low effective-code count plus a high top-code share indicates collapse.
- Reconstruction quality validates the codec, not Step 1 predictability. Step 1 must still be evaluated with anchor-token CE, decoded anchor quality, and generated-prefix rollouts.